# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset package using the `mlcroissant` library. All schema entities (record sets, fields, columns) are referenced by their unique `@id` values for traceability, integrity, and reproducibility.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
# Access metadata as object
metadata = ds.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Record sets encapsulate tabular data. Fields represent concepts, and columns are the actual values. All are referenced by their `@id`. Let's list the available record sets.

In [ ]:
# List available record sets and their @id
record_sets = ds.metadata.record_sets
print("Record Sets (@id):")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name','')}")

# Show fields and columns in each record set
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name','')})")
    fields = rs.get('fields', [])
    columns = rs.get('columns', [])
    if fields:
        print("  Fields (@id):")
        for f in fields:
            print(f"    - {f['@id']}: {f.get('name','')} [{f.get('dataType','')}]")
    if columns:
        print("  Columns (@id):")
        for c in columns:
            print(f"    - {c['@id']}: {c.get('name','')} [{c.get('dataType','')}]")

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for further analysis. All variables use exact `@id` values. Let's collect the record sets' `@id`.

In [ ]:
# Collect record set @id values
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    records = list(ds.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Print schema info for each record set
for rs_id in record_set_ids:
    print(f"\nColumns for Record Set {rs_id}:")
    print(dataframes[rs_id].columns.tolist())
    print(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes outlier removal, transformation, and grouping by key attributes.

For demonstration, we'll use the first record set and one of its numeric columns for filtering and normalization, referencing it by its `@id`.

In [ ]:
# Example: Select record set and field to analyze
# Find a numeric column in the first record set
rs_id = record_set_ids[0]
df = dataframes[rs_id]

# Find numeric columns from metadata
rs_meta = next((rs for rs in record_sets if rs['@id']==rs_id), None)
numeric_column_id = None
if rs_meta and 'columns' in rs_meta:
    for col in rs_meta['columns']:
        dtype = col.get('dataType','')
        if dtype in ['schema:Integer','schema:Float','schema:Number']:
            numeric_column_id = col['@id']
            break
if numeric_column_id and numeric_column_id in df.columns:
    # Filtering: use threshold
    threshold = df[numeric_column_id].mean()
    filtered_df = df[df[numeric_column_id] > threshold]
    print(f"Filtered records with {numeric_column_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    col_norm = f"{numeric_column_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_column_id] - filtered_df[numeric_column_id].mean()) / filtered_df[numeric_column_id].std()
    print(f"Normalized {numeric_column_id} for filtered records:")
    print(filtered_df[[numeric_column_id, col_norm]].head())

    # Try grouping by a categorical column
    group_field_id = None
    for col in rs_meta['columns']:
        dtype = col.get('dataType','')
        if dtype in ['schema:Text','schema:Boolean'] and col['@id'] != numeric_column_id:
            group_field_id = col['@id']
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_column_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_column_id}):")
        print(grouped_df.head())
else:
    print("No numeric column detected for EDA in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Reference axes and legend with `@id` for clarity.

Example: Histogram and boxplot for the selected numeric column.

In [ ]:
if numeric_column_id and numeric_column_id in df.columns:
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    df[numeric_column_id].hist(bins=10)
    plt.title(f"Distribution of {numeric_column_id}")
    plt.xlabel(numeric_column_id)
    plt.ylabel("Count")
    plt.subplot(1,2,2)
    df.boxplot(column=numeric_column_id)
    plt.title(f"Boxplot of {numeric_column_id}")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric column available for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset package via Croissant schema, explored its metadata and tabular structure by referencing all entities by their `@id`, and performed basic exploratory analysis using `mlcroissant`.
- **Data integrity**: Maintained by referencing all entities via `@id`.
- **Analysis**: Demonstrated filtering, normalization, and grouping on numeric fields.
- **Visualization**: Summarized distributions of key columns.

Further exploration could include advanced statistical analysis and integration with additional clinical or genetic datasets.